In [1]:
pip install pandas numpy google-cloud-bigquery pandas-gbq prophet

   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.3 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.3 MB 645.7 kB/s eta 0:00:08
   --- ------------------------------------ 0.5/5.3 MB 645.7 kB/s eta 0:00:08
   ----- ---------------------------------- 0.8/5.3 MB 657.8 kB/s eta 0:00:07
   ----- ---------------------------------- 0.8/5.3 MB 657.8 kB/s eta 0:00:07
   ------- -------------------------------- 1.0/5.3 MB 653.7 kB/s eta 0:00:07
   --------- ------------------------------ 1.3/5.3 MB 684.9 kB/s eta 0:00:06
   --------- ------------------------------ 1.3/5.3 MB 684.9 kB/s eta 0:00:06
   ----------- ---------------------------- 1.6/5.3 MB 682.0 kB/s eta 0:00:06
   ----------- ---------------------------- 1.6/5.3 MB 682.0 kB/s eta 0:00:06
   ------------- -----

In [ ]:
import pandas as pd
import pandas_gbq

# 1. The SQL Query
query = """
SELECT 
    product_sk,
    CAST(order_timestamp AS DATE) AS sale_date,
    SUM(fulfilled_qty) AS daily_units_sold
FROM `mart.fact_sales`
GROUP BY 
    product_sk, 
    sale_date
ORDER BY 
    sale_date ASC
"""

# project ID beolow (removed it so no one sees it)
project_id = 'your_google_project_id_here'

print("Knocking on BigQuery's door... Downloading data...")

df = pandas_gbq.read_gbq(query, project_id=project_id)

print("Download complete!")

df.head()

Knocking on BigQuery's door... Downloading data...
Downloading: 100%|██████████|
Download complete!


,product_sk,sale_date,daily_units_sold
0,6fe40271cac33195ca681e3512149854,2025-01-01,2
1,14f41f211d61656a7da8380085260840,2025-01-01,11
2,eecfa1f84d127327269b5bb7a9823da7,2025-01-01,12
3,8f20d19529c6be0b467ea535b5edff25,2025-01-01,6
4,5f7c380324275480645e09f4e0b1c3b2,2025-01-01,3


In [18]:
from prophet import Prophet
import warnings
import logging

# ==========================================================
# 1. SHUT THE AI UP
# The AI talks too much while it works and prints out 
# thousands of useless warning messages. We turn this off 
# so it doesn't freeze our computer screen.
# ==========================================================
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)

# ==========================================================
# 2. RENAME COLUMNS FOR THE PICKY AI
# The AI refuses to read columns named 'sale_date' or 
# 'daily_units_sold'. It strictly demands the date be called 
# 'ds' and the numbers be called 'y'. 
# ==========================================================
prophet_df = df.rename(columns={'sale_date': 'ds', 'daily_units_sold': 'y'})

# ==========================================================
# 3. CREATE AN EMPTY BUCKET
# We are going to predict one product at a time. 
# We need an empty bucket to throw the answers into as we 
# finish each one.
# ==========================================================
all_predictions = []

# ==========================================================
# 4. GET THE GROCERY LIST OF PRODUCTS
# The AI gets confused if you hand it every product at the 
# same time. We need a list of unique products so we can feed 
# them to the AI one by one.
# ==========================================================
unique_products = prophet_df['product_sk'].unique()
print(f"Starting AI forecast for {len(unique_products)} products. This might take a minute...")

for sku in unique_products:
    
    # Grab only the past sales history for THIS specific product.
    sku_history = prophet_df[prophet_df['product_sk'] == sku].copy()
    
    # ==========================================================
    # 5. SKIP NEW PRODUCTS
    # If a product has only been sold for 1 day, the AI cannot 
    # find a pattern. It needs at least 2 days of history, 
    # otherwise the math crashes. 
    # ==========================================================
    if len(sku_history) < 2:
        continue
        
    # ==========================================================
    # 6. TURN ON THE AI BRAIN
    # We turn off 'yearly_seasonality' (like Christmas spikes) 
    # because we don't have multiple years of data yet. If we 
    # leave it on, the AI gets confused.
    # ==========================================================
    model = Prophet(daily_seasonality=False, yearly_seasonality=False) 
    
    # ==========================================================
    # 7. TEACH THE AI
    # We hand the AI the past sales data for this product so 
    # it can study the pattern.
    # ==========================================================
    model.fit(sku_history)
    
    # ==========================================================
    # 8. GET A BLANK CALENDAR
    # The AI needs a blank piece of paper to write its guesses 
    # on. We generate a blank calendar for the next 30 days.
    # ==========================================================
    future_calendar = model.make_future_dataframe(periods=30)
    
    # ==========================================================
    # 9. MAKE THE GUESS
    # We force the AI to look at the blank calendar and fill 
    # in the future sales.
    # ==========================================================
    forecast = model.predict(future_calendar)
    
    # ==========================================================
    # 10. THROW AWAY THE JUNK MATH
    # The AI spits out 20 columns of crazy statistical math we 
    # don't care about. We only keep the Date ('ds') and the 
    # AI's predicted guess ('yhat').
    # ==========================================================
    forecast = forecast[['ds', 'yhat']].copy()
    
    # Put the product name back on the paper so we know what product this is.
    forecast['product_sk'] = sku  
    
    # Throw this finished prediction into our empty bucket.
    all_predictions.append(forecast)

# ==========================================================
# 11. DUMP THE BUCKET ONTO THE TABLE
# Take all the hundreds of separate product predictions inside 
# our bucket and glue them together into one giant spreadsheet.
# ==========================================================
final_forecast_df = pd.concat(all_predictions)

# ==========================================================
# 12. RENAME COLUMNS BACK TO NORMAL ENGLISH
# Change the weird AI names ('ds' and 'yhat') back to normal 
# words for BigQuery.
# ==========================================================
final_forecast_df = final_forecast_df.rename(columns={'ds': 'forecast_date', 'yhat': 'predicted_sales_qty'})

# ==========================================================
# 13. FIX IMPOSSIBLE AI MATH (REAL WORLD CHECK)
# The AI is just a math calculator. If a product is dying, 
# it might guess we will sell "-5" boxes. You cannot sell 
# negative boxes. `.clip(lower=0)` forces any negative guess 
# to be a zero.
#
# You also cannot sell "1.5" physical boxes. We force it to 
# round up or down to a whole box using `.round()`.
# ==========================================================
final_forecast_df['predicted_sales_qty'] = final_forecast_df['predicted_sales_qty'].clip(lower=0)
final_forecast_df['predicted_sales_qty'] = final_forecast_df['predicted_sales_qty'].round()

print("AI Forecasting Complete! 🔮")
final_forecast_df.tail(10)

12:00:18 - cmdstanpy - INFO - Chain [1] start processing


Starting AI forecast for 185 products. This might take a minute...


12:00:18 - cmdstanpy - INFO - Chain [1] done processing
12:00:18 - cmdstanpy - INFO - Chain [1] start processing
12:00:18 - cmdstanpy - INFO - Chain [1] done processing
12:00:19 - cmdstanpy - INFO - Chain [1] start processing
12:00:19 - cmdstanpy - INFO - Chain [1] done processing
12:00:19 - cmdstanpy - INFO - Chain [1] start processing
12:00:19 - cmdstanpy - INFO - Chain [1] done processing
12:00:19 - cmdstanpy - INFO - Chain [1] start processing
12:00:19 - cmdstanpy - INFO - Chain [1] done processing
12:00:20 - cmdstanpy - INFO - Chain [1] start processing
12:00:20 - cmdstanpy - INFO - Chain [1] done processing
12:00:20 - cmdstanpy - INFO - Chain [1] start processing
12:00:20 - cmdstanpy - INFO - Chain [1] done processing
12:00:20 - cmdstanpy - INFO - Chain [1] start processing
12:00:20 - cmdstanpy - INFO - Chain [1] done processing
12:00:21 - cmdstanpy - INFO - Chain [1] start processing
12:00:21 - cmdstanpy - INFO - Chain [1] done processing
12:00:21 - cmdstanpy - INFO - Chain [1] 

AI Forecasting Complete! 🔮


,forecast_date,predicted_sales_qty,product_sk
33,2026-07-19,8.0,dc45fabccd0a15c69d5e543780e7a8d1
34,2026-07-20,18.0,dc45fabccd0a15c69d5e543780e7a8d1
35,2026-07-21,10.0,dc45fabccd0a15c69d5e543780e7a8d1
36,2026-07-22,0.0,dc45fabccd0a15c69d5e543780e7a8d1
37,2026-07-23,8.0,dc45fabccd0a15c69d5e543780e7a8d1
38,2026-07-24,10.0,dc45fabccd0a15c69d5e543780e7a8d1
39,2026-07-25,9.0,dc45fabccd0a15c69d5e543780e7a8d1
40,2026-07-26,9.0,dc45fabccd0a15c69d5e543780e7a8d1
41,2026-07-27,18.0,dc45fabccd0a15c69d5e543780e7a8d1
42,2026-07-28,10.0,dc45fabccd0a15c69d5e543780e7a8d1


In [19]:
# ==========================================================
# STATION 3: THE SHIPPING ROOM
# The AI has finished. Now we need to push this brand new
# "future" spreadsheet back up into our Google BigQuery 
# database so our BI dashboard can read it.
# ==========================================================
print("Uploading AI predictions to BigQuery...")

# The name of the new table we are creating in BigQuery
destination_table = 'mart.ai_sales_forecast'

# Push the dataframe up to the cloud!
import pandas_gbq

pandas_gbq.to_gbq(
    dataframe=final_forecast_df,
    destination_table=destination_table,
    project_id=project_id,
    if_exists='replace'
    # If we run this script again tomorrow, overwrite the old predictions
)

print("Upload complete! The future is now in your database. 🚀")

Uploading AI predictions to BigQuery...


100%|██████████| 1/1 [00:00<?, ?it/s]

Upload complete! The future is now in your database. 🚀


# 🔮 AI Demand Forecasting Engine

**Author:** [AFTAB KHAN]

**Purpose:** Pulls historical daily sales from BigQuery `mart.fact_sales`, feeds the data into Meta's Prophet Machine Learning algorithm, and generates a 30-day future sales prediction. The AI forecast is then pushed back into BigQuery as `mart.ai_sales_forecast`.